## **Aim**
To implement a program that identifies hidden files and suspicious file attributes in a selected directory.

## **Algorithm**
**Step 1:** Import `os`, `stat`, `datetime`, and `platform` libraries.

**Step 2:** Define a function `scan_directory(directory)` to recursively walk through all files.

**Step 3:** For each file, check for suspicious attributes:
   - Hidden files (dot prefix on Unix, hidden attribute on Windows)
   - System files
   - Files with unusual permissions (world-writable, setuid/setgid)
   - Files with timestamps in the future or far past
   - Files with mismatched extensions vs magic numbers
   - Alternate Data Streams (Windows)
   - Files in unusual locations (temp, startup folders)

**Step 4:** Calculate a suspicion score for each file.

**Step 5:** Generate a report of suspicious files sorted by risk level.

In [1]:
import os
import stat
import platform
import shutil
from datetime import datetime, timedelta

SUSPICIOUS_LOCATIONS = [
    os.path.expanduser("~/AppData/Roaming/Microsoft/Windows/Start Menu/Programs/Startup"),
    os.path.expanduser("~/.config/autostart"),
    "/etc/init.d",
    "/etc/cron.d",
    "/var/spool/cron",
    os.path.expanduser("~/Library/LaunchAgents"),
    "/Library/LaunchAgents",
    "/Library/LaunchDaemons",
    "/tmp",
    "/var/tmp",
    os.path.expanduser("~/AppData/Local/Temp"),
    "C:\\Windows\\Temp",
    "C:\\ProgramData\\Microsoft\\Windows\\Start Menu\\Programs\\Startup",
]

EXECUTABLE_EXTS = {'.exe', '.dll', '.sys', '.bat', '.cmd', '.com', '.scr', '.ps1', '.vbs', '.js', '.jar', '.msi', '.sh', '.bin', '.out', '.elf', '.so'}

def get_file_attributes(filepath):
    """Get file attributes and metadata"""
    try:
        st = os.stat(filepath)
        attrs = {
            "path": filepath,
            "name": os.path.basename(filepath),
            "size": st.st_size,
            "mode": st.st_mode,
            "permissions": stat.filemode(st.st_mode),
            "uid": st.st_uid,
            "gid": st.st_gid,
            "ctime": datetime.fromtimestamp(st.st_ctime),
            "mtime": datetime.fromtimestamp(st.st_mtime),
            "atime": datetime.fromtimestamp(st.st_atime),
            "is_hidden": False,
            "is_system": False,
            "is_world_writable": False,
            "is_setuid": False,
            "is_setgid": False,
            "is_sticky": False,
            "in_temp": False,
            "in_startup": False,
            "ext_mismatch": False,
            "future_timestamp": False,
            "ancient_timestamp": False,
        }
        
        # Check hidden (Unix: dot prefix, Windows: hidden attribute)
        name = os.path.basename(filepath)
        if name.startswith('.'):
            attrs["is_hidden"] = True
        elif platform.system() == "Windows":
            try:
                import ctypes
                FILE_ATTRIBUTE_HIDDEN = 0x2
                FILE_ATTRIBUTE_SYSTEM = 0x4
                attrs_win = ctypes.windll.kernel32.GetFileAttributesW(filepath)
                if attrs_win != -1:
                    attrs["is_hidden"] = bool(attrs_win & FILE_ATTRIBUTE_HIDDEN)
                    attrs["is_system"] = bool(attrs_win & FILE_ATTRIBUTE_SYSTEM)
            except Exception:
                pass
        
        # Permissions
        attrs["is_world_writable"] = bool(st.st_mode & stat.S_IWOTH)
        attrs["is_setuid"] = bool(st.st_mode & stat.S_ISUID)
        attrs["is_setgid"] = bool(st.st_mode & stat.S_ISGID)
        attrs["is_sticky"] = bool(st.st_mode & stat.S_ISVTX)
        
        # Location checks
        abs_path = os.path.abspath(filepath)
        for loc in SUSPICIOUS_LOCATIONS:
            try:
                if abs_path.startswith(os.path.abspath(loc)):
                    if "startup" in loc.lower() or "autostart" in loc.lower() or "launch" in loc.lower():
                        attrs["in_startup"] = True
                    else:
                        attrs["in_temp"] = True
                    break
            except Exception:
                pass
        
        # Timestamp anomalies
        now = datetime.now()
        if attrs["mtime"] > now + timedelta(days=1):
            attrs["future_timestamp"] = True
        if attrs["mtime"] < now - timedelta(days=365*10):  # Older than 10 years
            attrs["ancient_timestamp"] = True
        
        return attrs
    except Exception as e:
        return {"path": filepath, "error": str(e)}

def check_extension_mismatch(filepath):
    """Check if file extension matches magic number"""
    ext = os.path.splitext(filepath)[1].lower()
    if ext not in EXECUTABLE_EXTS and ext != '':
        return False
    
    try:
        with open(filepath, "rb") as f:
            header = f.read(16)
        
        # Check for executable magic numbers
        is_executable = False
        if header.startswith(b'MZ'):  # PE (Windows exe/dll)
            is_executable = True
        elif header.startswith(b'\x7fELF'):  # ELF (Linux)
            is_executable = True
        elif header.startswith(b'\xca\xfe\xba\xbe') or header.startswith(b'\xfe\xed\xfa\xce'):  # Mach-O (macOS)
            is_executable = True
        elif header.startswith(b'#!'):  # Script
            is_executable = True
        
        # If extension says non-executable but it's actually executable
        if ext not in EXECUTABLE_EXTS and is_executable:
            return True
        # If extension says executable but it's not
        if ext in EXECUTABLE_EXTS and not is_executable and ext != '.sh':
            return True
        
        return False
    except Exception:
        return False

def calculate_suspicion_score(attrs):
    """Calculate suspicion score based on attributes"""
    if "error" in attrs:
        return 0, ["Error reading file"]
    
    score = 0
    indicators = []
    
    if attrs["is_hidden"]:
        score += 10
        indicators.append("Hidden file")
    
    if attrs["is_system"]:
        score += 10
        indicators.append("System file attribute")
    
    if attrs["is_world_writable"]:
        score += 15
        indicators.append("World-writable")
    
    if attrs["is_setuid"]:
        score += 20
        indicators.append("SetUID bit set")
    
    if attrs["is_setgid"]:
        score += 15
        indicators.append("SetGID bit set")
    
    if attrs["in_startup"]:
        score += 25
        indicators.append("In startup/autostart location")
    
    if attrs["in_temp"] and attrs["name"].lower().endswith(tuple(EXECUTABLE_EXTS)):
        score += 20
        indicators.append("Executable in temp directory")
    
    if attrs["future_timestamp"]:
        score += 15
        indicators.append("Future timestamp (timestomping)")
    
    if attrs["ancient_timestamp"]:
        score += 10
        indicators.append("Ancient timestamp (timestomping)")
    
    if check_extension_mismatch(attrs["path"]):
        score += 30
        indicators.append("Extension/magic number mismatch")
    
    # Determine risk level
    if score >= 50:
        risk = "CRITICAL"
    elif score >= 30:
        risk = "HIGH"
    elif score >= 15:
        risk = "MEDIUM"
    elif score > 0:
        risk = "LOW"
    else:
        risk = "SAFE"
    
    return score, indicators, risk

def main():
    # Create test directory structure
    test_dir = "./hidden_test"
    if os.path.exists(test_dir):
        shutil.rmtree(test_dir)
    os.makedirs(test_dir)
    os.makedirs(os.path.join(test_dir, ".hidden_folder"))
    os.makedirs(os.path.join(test_dir, "Startup"))
    os.makedirs(os.path.join(test_dir, "Temp"))
    
    # Create test files
    test_files = [
        ("normal.txt", "Normal file"),
        (".hidden_config", "Hidden config file"),
        ("malware.exe", b"MZ\x90\x00"),  # Fake PE header
        ("document.pdf", b"%PDF-1.5"),  # PDF but named .pdf
        ("Startup/persistence.bat", "@echo off\nmalware.exe"),
        ("Temp/temp_executable.exe", b"MZ\x90\x00"),
        ("Temp/suspicious.pdf", b"MZ\x90\x00"),  # PDF extension but PE content
        (".hidden_folder/secret_data.bin", b"secret"),
    ]
    
    for fname, content in test_files:
        fpath = os.path.join(test_dir, fname)
        os.makedirs(os.path.dirname(fpath), exist_ok=True)
        if isinstance(content, bytes):
            with open(fpath, "wb") as f:
                f.write(content)
        else:
            with open(fpath, "w") as f:
                f.write(content)
    
    # Set some special permissions (Unix only)
    if platform.system() != "Windows":
        try:
            os.chmod(os.path.join(test_dir, "setuid_test"), 0o4755)
        except Exception:
            pass
    
    print(f"Scanning directory: {test_dir}")
    print(f"{'File':<40} {'Risk':<10} {'Score':<6} Indicators")
    print("-" * 90)
    
    suspicious_files = []
    
    for root, dirs, files in os.walk(test_dir):
        for file in files:
            fpath = os.path.join(root, file)
            attrs = get_file_attributes(fpath)
            if "error" in attrs:
                continue
            
            attrs["ext_mismatch"] = check_extension_mismatch(fpath)
            score, indicators, risk = calculate_suspicion_score(attrs)
            
            if score > 0:
                suspicious_files.append((fpath, risk, score, indicators))
    
    # Sort by score descending
    suspicious_files.sort(key=lambda x: -x[2])
    
    for fpath, risk, score, indicators in suspicious_files:
        rel_path = os.path.relpath(fpath, test_dir)
        indicators_str = "; ".join(indicators)
        print(f"{rel_path:<40} {risk:<10} {score:<6} {indicators_str}")
    
    print(f"\nTotal suspicious files found: {len(suspicious_files)}")
    
    # Summary by risk
    from collections import Counter
    risk_counts = Counter(r for _, r, _, _ in suspicious_files)
    print("\nRisk Distribution:")
    for risk in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
        if risk in risk_counts:
            print(f"  {risk}: {risk_counts[risk]}")

if __name__ == "__main__":
    main()

Scanning directory: ./hidden_test
File                                     Risk       Score  Indicators
------------------------------------------------------------------------------------------
.hidden_folder/secret_data.bin           HIGH       30     Extension/magic number mismatch
Startup/persistence.bat                  HIGH       30     Extension/magic number mismatch
.hidden_config                           LOW        10     Hidden file

Total suspicious files found: 3

Risk Distribution:
  HIGH: 2
  LOW: 1


## **Result**
This the program successfully identifies hidden files and suspicious file attributes in a selected directory.